In [ ]:
# ============================================================
# Download PLUG corpus and extract sad Ukrainian sentences
# ============================================================

!pip -q install stanza pandas tqdm gitpython

import re
import subprocess
from pathlib import Path

import pandas as pd
import stanza
from tqdm import tqdm

# ------------------------------------------------------------
# Clone repository
# ------------------------------------------------------------

REPO = "https://github.com/Dandelliony/pluperfect_grac.git"
ROOT = Path("pluperfect_grac")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO], check=True)

# ------------------------------------------------------------
# Download Ukrainian tokenizer
# ------------------------------------------------------------

stanza.download("uk")

nlp = stanza.Pipeline(
    lang="uk",
    processors="tokenize",
    use_gpu=False,
    verbose=False
)

# ------------------------------------------------------------
# Find text files
# ------------------------------------------------------------

files = list(ROOT.rglob("*.txt"))

print(f"Found {len(files)} text files.")

# ------------------------------------------------------------
# Sadness lexicon
# ------------------------------------------------------------

SAD_WORDS = {

    "горе","горя","горем",

    "туга","туги",

    "печаль","печалі",

    "смуток","смутку",

    "сум","сумно","сумний","сумна",

    "журба",

    "сльоза","сльози",

    "ридати","ридала","ридали",

    "плакати","плакала","плач",

    "мука","муки",

    "страждання","страждати",

    "болить","боліло","болючий",

    "самота","самотній","самотня",

    "нещасний","нещасна",

    "безталанний","безталанна",

    "покинутий","покинута",

    "втрата","втратив","втратила",

    "розпач",

    "відчай",

    "біда",

    "тяжко",

    "важко",

    "помер",

    "умер",

    "смерть",

    "могила"
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

word_pattern = re.compile(r"[а-щьюяґєії'-]+")

def sadness_score(text):

    words = word_pattern.findall(text.lower())

    return sum(w in SAD_WORDS for w in words)


In [ ]:
import random

files_experiment = random.sample(files, 1000)

In [ ]:
# ------------------------------------------------------------
# Extract sentences
# ------------------------------------------------------------

rows = []

for file in tqdm(files_experiment):

    try:
        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except:
        continue

    if len(text) < 100:
        continue

    doc = nlp(text)

    for sent in doc.sentences:

        sentence = sent.text.strip()

        if len(sentence) < 40:
            continue

        score = sadness_score(sentence)

        if score >= 2:

            rows.append({

                "score": score,

                "sentence": sentence,

                "source_file": file.name,

                "path": str(file)

            })

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

df = (
    pd.DataFrame(rows)
      .sort_values("score", ascending=False)
      .drop_duplicates("sentence")
      .reset_index(drop=True)
)

print(f"\nSad sentences found: {len(df):,}")

df.to_csv(
    "sad_sentences.csv",
    index=False,
    encoding="utf-8"
)

display(df.head(30))

print("\nSaved to sad_sentences.csv")

In [ ]:
# ============================================================
# Generate modern Ukrainian variants — direct API calls
# No batches, no waiting. Runs top to bottom.
# Checkpoint-safe: can be re-run if interrupted.
# ============================================================

!pip -q install openai pandas tqdm

import json, os, re, time, uuid
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ── Config ──────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file")
MODEL          = "gpt-4.1-mini"   # fast and cheap; change to "gpt-4.1" for higher quality
MAX_ROWS       = 100              # set to None for all rows
N_VARIANTS     = 3               # modern variants per sentence
INPUT_CSV      = "sad_sentences.csv"
OUTPUT_CSV     = "modernized_training_pairs_flat.csv"
CHECKPOINT_DIR = Path("generation_checkpoints")

client = OpenAI(api_key=OPENAI_API_KEY)
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Prompts ─────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an expert in Ukrainian language and classical literature. "
    "Rewrite excerpts from classical Ukrainian literature into natural modern Ukrainian. "
    "Preserve meaning and emotion. Change archaic words and syntax to contemporary phrasing. "
    "Do NOT imitate classical style. Do NOT write Russian or English. Do NOT explain. "
    "Return only valid JSON."
)

def user_prompt(classic_text, n=N_VARIANTS):
    return (
        f"Classical Ukrainian excerpt:\n\n{classic_text}\n\n"
        f"Generate {n} modern Ukrainian variants that sound like something "
        f"a Ukrainian person in 2026 would write in social networks, messages or diaries.\n\n"
        f"Return JSON:\n"
        f'{{ "keep": true, "emotion": "sadness|loneliness|grief|despair|exhaustion|pain|nostalgia|other", '
        f'"modern_variants": ["...", "...", "..."] }}'
    )

# ── Load sentences ───────────────────────────────────────────
try:
    df
    print(f"Using df from memory ({len(df)} rows)")
except NameError:
    df = pd.read_csv(INPUT_CSV)
    print(f"Loaded {len(df)} sentences from {INPUT_CSV}")

df = df.dropna(subset=["sentence"]).drop_duplicates("sentence").reset_index(drop=True)
df_work = df.head(MAX_ROWS).copy() if MAX_ROWS else df.copy()
print(f"Processing {len(df_work)} sentences x {N_VARIANTS} variants = up to {len(df_work)*N_VARIANTS} pairs")

# ── Resume from checkpoint ───────────────────────────────────
done_ids = set()
all_pairs = []
ckpt_file = CHECKPOINT_DIR / "pairs_checkpoint.jsonl"
if ckpt_file.exists():
    with open(ckpt_file, encoding="utf-8") as fh:
        for line in fh:
            row = json.loads(line)
            all_pairs.append(row)
            done_ids.add(row["row_idx"])
    print(f"Resuming — {len(done_ids)} rows already done")

rows_todo = df_work[~df_work.index.isin(done_ids)]
print(f"Rows left to process: {len(rows_todo)}")

# ── API call with retry ──────────────────────────────────────
def call_api(sentence, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt(sentence)},
                ],
                temperature=0.8,
                max_tokens=600,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

# ── Main generation loop ─────────────────────────────────────
with open(ckpt_file, "a", encoding="utf-8") as ckpt_f:
    for row_idx, row in tqdm(rows_todo.iterrows(), total=len(rows_todo), desc="Generating"):
        sentence = str(row["sentence"]).strip()
        result = call_api(sentence)
        if result is None or not result.get("keep", True):
            continue
        for i, variant in enumerate(result.get("modern_variants", [])[:N_VARIANTS]):
            if not variant.strip():
                continue
            pair = {
                "row_idx":        int(row_idx),
                "model":          MODEL,
                "classic_target": sentence,
                "modern_input":   variant.strip(),
                "emotion":        result.get("emotion", "other"),
                "source_file":    str(row.get("source_file", "")),
                "variant_id":     i,
            }
            all_pairs.append(pair)
            ckpt_f.write(json.dumps(pair, ensure_ascii=False) + "\n")
        time.sleep(0.05)

# ── Save flat CSV ────────────────────────────────────────────
flat_df = pd.DataFrame(all_pairs)
flat_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"\nDone! {len(flat_df)} training pairs saved to {OUTPUT_CSV}")
print(flat_df[["classic_target", "modern_input", "emotion"]].head(6).to_string())

In [ ]:
# ============================================================
# Inspect results and build SFT training files
# ============================================================

import json, uuid, random
import pandas as pd
from pathlib import Path
from IPython.display import display

OUTPUT_CSV = "modernized_training_pairs_flat.csv"

flat_df = pd.read_csv(OUTPUT_CSV)
print(f"Total pairs: {len(flat_df)}")
print(f"Unique sentences: {flat_df['classic_target'].nunique()}")
print("\nEmotion distribution:")
print(flat_df["emotion"].value_counts().to_string())
print()
display(flat_df[["classic_target", "modern_input", "emotion"]].head(8))

# ── master_dataset.jsonl ─────────────────────────────────────
Path("data/processed").mkdir(parents=True, exist_ok=True)
master_path = Path("data/processed/master_dataset.jsonl")
with open(master_path, "w", encoding="utf-8") as f:
    for _, row in flat_df.iterrows():
        record = {
            "id":             f"plug_{str(uuid.uuid4())[:8]}_v{row['variant_id']}",
            "classic_target": str(row["classic_target"]),
            "modern_input":   str(row["modern_input"]),
            "emotion":        str(row.get("emotion", "other")),
            "source_file":    str(row.get("source_file", "")),
            "author":         "",
            "source_dataset": "PluG",
            "quality_score":  4,
            "split":          "train",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"master_dataset.jsonl saved — {len(flat_df)} rows")

# ── SFT chat format splits ───────────────────────────────────
Path("data/sft").mkdir(parents=True, exist_ok=True)
INSTRUCTION = "Перепиши сучасний ukrainian текст у стилі української класичної літератури."

pairs = flat_df.to_dict("records")
random.shuffle(pairs)
n       = len(pairs)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
splits  = {
    "train": pairs[:n_train],
    "val":   pairs[n_train: n_train + n_val],
    "test":  pairs[n_train + n_val:],
}

for split_name, rows in splits.items():
    out = Path(f"data/sft/{split_name}.jsonl")
    with open(out, "w", encoding="utf-8") as f:
        for r in rows:
            msg = {
                "messages": [
                    {"role": "user",      "content": f"{INSTRUCTION}\n\nТекст: {r['modern_input']}"},
                    {"role": "assistant", "content": r["classic_target"]},
                ]
            }
            f.write(json.dumps(msg, ensure_ascii=False) + "\n")
    print(f"  {split_name}: {len(rows)} examples -> data/sft/{split_name}.jsonl")

print("\nSFT dataset ready. Proceed to Cell 13 to fine-tune with MLX.")

## Next Steps

Continued in cells below.

## Step 5: Quality Filter

Filter the master dataset to remove low-quality, contaminated, or duplicate pairs.

In [ ]:
import sys
sys.path.insert(0, ".")

from pathlib import Path

# If master_dataset.jsonl exists, run filter
master_path = Path("data/processed/master_dataset.jsonl")

if master_path.exists():
    from src.data.quality_filter import run_filter
    filtered = run_filter()
    print(f"Filtered dataset: {len(filtered)} pairs")
else:
    print("Master dataset not found. Run Step 4 (generate_modern_variants) first.")
    print("Or create a sample from the existing modernized_training_pairs_flat.csv...")

    # Fallback: convert existing CSV from pilot run
    import pandas as pd, json, uuid
    csv_path = Path("modernized_training_pairs_flat.csv")
    if csv_path.exists():
        df_pairs = pd.read_csv(csv_path)
        master_path.parent.mkdir(parents=True, exist_ok=True)
        with open(master_path, "w", encoding="utf-8") as f:
            for _, row in df_pairs.iterrows():
                record = {
                    "id": f"plug_{str(uuid.uuid4())[:8]}_v0",
                    "classic_target": row.get("classic_target", row.get("sentence", "")),
                    "modern_input": row.get("modern_input", row.get("variant", "")),
                    "emotion": row.get("emotion", "other"),
                    "source_file": row.get("source_file", "unknown"),
                    "author": row.get("author", ""),
                    "work_title": "",
                    "source_dataset": "PluG",
                    "quality_score": 4,
                    "split": "train"
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Converted {len(df_pairs)} rows to master_dataset.jsonl")
        from src.data.quality_filter import run_filter
        filtered = run_filter()
        print(f"Filtered: {len(filtered)} pairs")

## Step 6: Build SFT Dataset

Convert master dataset to instruction-following chat format.

In [ ]:
from src.data.build_sft_dataset import build_sft_dataset
from pathlib import Path

filtered_path = Path("data/processed/master_dataset_filtered.jsonl")
if filtered_path.exists():
    build_sft_dataset(source=filtered_path)
    for split in ["train", "val", "test"]:
        p = Path(f"data/sft/{split}.jsonl")
        if p.exists():
            n = sum(1 for _ in open(p, encoding="utf-8") if _.strip())
            print(f"  {split}: {n} examples")
else:
    print("Filtered dataset not found. Run Step 5 first.")

## Step 7: Preview SFT Examples

Check the final training format.

In [ ]:
import json
from pathlib import Path

train_path = Path("data/sft/train.jsonl")
if train_path.exists():
    with open(train_path, encoding="utf-8") as f:
        examples = [json.loads(l) for l in f if l.strip()][:3]
    for i, ex in enumerate(examples, 1):
        print(f"=== Example {i} ===")
        for msg in ex["messages"]:
            print(f"[{msg["role"]}] {msg["content"][:200]}")
        print()
else:
    print("SFT dataset not found. Run Step 6 first.")

## Step 8: Fine-Tune with MLX-LM

Run LoRA fine-tuning on Apple Silicon. Requires mlx-lm installed.

In [ ]:
import os, sys
from pathlib import Path

# ── Local training settings (override Docker .env values) ──
os.environ.pop("MLFLOW_TRACKING_URI", None)
os.environ["BASE_MODEL"] = "mlx-community/Qwen3-8B-4bit"
os.environ["MODEL_VERSION"] = "qwen3-8b-lora-v1"

sys.path.insert(0, str(Path(".").resolve()))

# ── Check prerequisites ──
if not Path("data/sft/train.jsonl").exists():
    raise FileNotFoundError("data/sft/train.jsonl not found. Run Cells 3 and 4 first.")

try:
    import mlx
    print("MLX available - Apple Silicon detected")
except ImportError:
    raise ImportError("MLX not installed. Run: pip install mlx-lm")

# ── Verify model is cached (you already downloaded it) ──
print(f"Base model: {os.environ['BASE_MODEL']}")
print("Model should load from local cache — no long download expected.")

# ── Run fine-tuning ──
import importlib
import src.training.train as train_mod
importlib.reload(train_mod)  # reload so CONFIG picks up env vars above
train_mod.main()

## Step 9: Evaluate

Run automatic metrics on golden prompts.

In [ ]:
os.environ["INFERENCE_MODE"] = "mock"  # change to "local" or "openai" after fine-tuning
os.environ["DATABASE_URL"] = "sqlite:///eval_test.db"
os.environ["STORE_USER_INPUTS"] = "false"
os.environ["REDIS_URL"] = "redis://localhost:6379/0"

from src.api.config import get_settings
get_settings.cache_clear()

from src.evaluation.run_eval import run_eval
run_eval(inference_mode="mock")